# Learned Orchestration: A Case Study of Sakana Fugu

The runnable code of [Learned Orchestration: A Case Study of Sakana Fugu](https://residual-stream.lvtan71.workers.dev/posts/learned-orchestration-sakana-fugu/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/learned-orchestration-sakana-fugu.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "learned-orchestration-sakana-fugu"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in []:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## The results, read critically

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/learned-orchestration-sakana-fugu/#the-results-read-critically)

In [ ]:
# Table 1 of the Sakana Fugu technical report (arXiv 2606.21228v2), as
# reported: Fugu-Ultra, Fugu, then the three workers both orchestrate.
# Baselines are provider-reported where available, so the comparison is
# not a controlled one. Margin = orchestrator minus the best single worker.
rows = {
    "SWE Bench Pro":          (73.7, 59.0, 69.2, 54.2, 58.6),
    "Terminal Bench 2.1":     (82.1, 80.2, 74.6, 70.3, 78.2),
    "LiveCodeBench":          (93.2, 92.9, 87.8, 88.5, 85.3),
    "LiveCodeBench Pro":      (90.8, 87.8, 84.8, 82.9, 88.4),
    "Humanity's Last Exam":   (50.0, 47.2, 49.8, 44.4, 41.4),
    "CharXiv Reasoning":      (86.6, 85.1, 84.2, 83.3, 84.1),
    "GPQA Diamond":           (95.5, 95.5, 92.0, 94.3, 93.6),
    "SciCode":                (58.7, 60.1, 53.5, 58.9, 56.1),
    "tau^3 Banking":          (20.6, 21.7, 20.6, 8.4, 20.6),
    "Long Context Reasoning": (73.3, 74.7, 67.7, 72.7, 74.3),
    "MRCRv2":                 (93.6, 86.6, 87.9, 84.9, 94.8),
}
workers = ("Claude Opus 4.8", "Gemini 3.1", "GPT-5.5")
print(f"{'benchmark':24s} {'best worker':>22s} {'Fugu-Ultra':>11s} {'Fugu':>7s}")
ahead = {"Fugu-Ultra": 0, "Fugu": 0}
for name, (ultra, fugu, *w) in rows.items():
    best = max(w)
    who = workers[w.index(best)]
    print(f"{name:24s} {who:>15s} {best:5.1f}  {ultra - best:+10.1f} {fugu - best:+7.1f}")
    ahead["Fugu-Ultra"] += ultra > best; ahead["Fugu"] += fugu > best
print(f"\nahead of the best single worker: Fugu-Ultra on {ahead['Fugu-Ultra']} of {len(rows)}, Fugu on {ahead['Fugu']} of {len(rows)}")
best_on = {}
for name, (_, _, *w) in rows.items():
    best_on.setdefault(workers[w.index(max(w))], []).append(name)
for k, v in best_on.items():
    print(f"  {k} is the best worker on {len(v)}: {', '.join(v)}")

The post printed:

```text
benchmark                           best worker  Fugu-Ultra    Fugu
SWE Bench Pro            Claude Opus 4.8  69.2        +4.5   -10.2
Terminal Bench 2.1               GPT-5.5  78.2        +3.9    +2.0
LiveCodeBench                 Gemini 3.1  88.5        +4.7    +4.4
LiveCodeBench Pro                GPT-5.5  88.4        +2.4    -0.6
Humanity's Last Exam     Claude Opus 4.8  49.8        +0.2    -2.6
CharXiv Reasoning        Claude Opus 4.8  84.2        +2.4    +0.9
GPQA Diamond                  Gemini 3.1  94.3        +1.2    +1.2
SciCode                       Gemini 3.1  58.9        -0.2    +1.2
tau^3 Banking            Claude Opus 4.8  20.6        +0.0    +1.1
Long Context Reasoning           GPT-5.5  74.3        -1.0    +0.4
MRCRv2                           GPT-5.5  94.8        -1.2    -8.2

ahead of the best single worker: Fugu-Ultra on 7 of 11, Fugu on 7 of 11
  Claude Opus 4.8 is the best worker on 4: SWE Bench Pro, Humanity's Last Exam, CharXiv Reasoning, tau^3 Banking
  GPT-5.5 is the best worker on 4: Terminal Bench 2.1, LiveCodeBench Pro, Long Context Reasoning, MRCRv2
  Gemini 3.1 is the best worker on 3: LiveCodeBench, GPQA Diamond, SciCode
```